# WiFi classification with PyTorch

Prepare the incremental NPY cache, hold out target domains and split source segments, create memory-mapped DataLoaders, and train the selected paired model. Only validation selects model weights; calibration and test remain reserved for later scoring and evaluation.


In [ ]:
# Set a YAML path here for interactive use; None uses experiments/baseline.yaml.
config_path = None
experiment_dir = None
project_root = None


In [ ]:
from pathlib import Path
import sys
import re
import json
import numpy as np
from scipy.io import loadmat, whosmat
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(project_root or Path.cwd()).expanduser().resolve()
if (PROJECT_ROOT / "step02_models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON_ROOT = PROJECT_ROOT / "python"
DATA_ROOT = PROJECT_ROOT / "data"
NPY_ROOT = PYTHON_ROOT / "dataset"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))
from step00_config.loader import load_config
config = load_config(config_path)


## Experiment settings

Choose `config_path` in the first cell, or leave it as `None` for `experiments/baseline.yaml`. The command-line runner injects a saved configuration and its output directory. The runner executes the complete notebook, including scoring and evaluation. When working interactively, you choose which cells to execute.

### Data and split settings

The template is shared by all three datasets. Brackets mark the optional repeat identifier. Choose `activity`, `user`, or `position` as the target; `p1` alone is only a placeholder, not a useful position-classification problem.

Select a folder under `python/dataset/`: for crossroom use `generated_features`; for self_time/setting_dataset use the dataset directory or a compatible subset. The template matches `_time.npy`; `_stft.npy` is paired automatically.

`data.datasets` in YAML accepts one or several directories relative to `python/dataset/` (or absolute cache paths); these become `data_config["roots"]`. Overlapping roots are deduplicated by resolved file path. All selected files must have matching feature shapes. Roots are pooled before stratified splitting; this is not a train-on-one-dataset/test-on-another protocol. The default uses crossroom/generated_features. `doc` is the shared label for document activity.


In [ ]:
from step00_config.schema import InferenceConfig
inference_batch_size = InferenceConfig(**config["inference"]).batch_size

data_config = {
    "domain_key": config["data"]["domain_key"],
    "roots": [NPY_ROOT / folder for folder in config["data"]["datasets"]],
    "filename_template": config["data"]["filename_template"],
    "task": config["data"]["task"],
}
split_seed = config["split_seed"]
training_seed = config["training_seed"]
split_config = config["split"].copy()
prepare_npy_cache = config["prepare_npy_cache"]
print(json.dumps(config, indent=2))


### Model and training settings

Edit `model`, `training` (including `training.scheduler`), `augmentation`, `sampling`, `encoder_task`, `encoder_background`, and `decoder` in the YAML file. These objects are created before DataLoaders and the model. For interactive experiments, you can also edit their attributes here. `training.optimizer_kwargs` passes optimizer-specific arguments such as SGD momentum.

`split_seed` fixes data splits and scorer CV folds; `training_seed` controls initialization, shuffle, Dropout and the independent augmentation RNG. The YAML selects which components are enabled. Cosine uses the epoch budget by default; plateau monitors domain-mean validation CE. Early stopping requires task CE patience and stable active DA/CL metrics after warmup. With auxiliary monitors, the earliest possible stop is warmup + 2 × window + patience − 1; without them, task patience alone applies.

Augmentation runs only on training batches. The master `augmentation.enabled` switch controls the whole component; `time_noise.enabled`, `time_mask.enabled`, and `stft_mask.enabled` select any combination. Gaussian/uniform noise is added to standardized time features; time masking then zeros a short interval across all rows/groups. STFT masks operate on reconstructed time/frequency axes. Time and STFT masks are sampled independently. `frequency_bins` is 32 for setting_dataset/self_time and 64 for crossroom. New noise and bands are sampled each call; the RNG is seeded once per training setup, not per epoch. No crops or shifts are applied.


`encoder_background.enabled` adds a separately initialized background encoder shared across source domains.
Each background objective has its own switch. Task CE/CL/domain adversarial are configured under encoder_task. Task CE has independent enabled/weight controls; raw training and validation CE remain reported. train_objective_loss records the combined weighted training objective.
Reconstruction targets the model inputs (standardized time, signed-log STFT), not raw CSI.
Background activity-head fitting and all encoder/decoder updates occur only in the training loop.


In [ ]:
from dataclasses import asdict
from step02_models.build import build_model
from step00_config.schema import ModelConfig, TrainingConfig, SchedulerConfig, AugmentationConfig, AdversarialConfig, ContrastiveConfig, SamplingConfig
from step03_training.monitoring import TrainingMonitor

from step00_config.schema import EncoderTaskConfig
encoder_task_config = EncoderTaskConfig(**config["encoder_task"])
model_config = ModelConfig(**config["model"])
training_config = TrainingConfig(**config["training"])
augmentation_config = AugmentationConfig(**config["augmentation"])
adversarial_config = AdversarialConfig(**encoder_task_config.adversarial)
contrastive_config = ContrastiveConfig(**encoder_task_config.contrastive)
sampling_config = SamplingConfig(**config["sampling"])

from step00_config.schema import DecoderConfig, EncoderBackgroundConfig
encoder_background_config = EncoderBackgroundConfig(**config.get("encoder_background", {}))

decoder_config = DecoderConfig(**config["decoder"])


### Monitoring settings

The YAML `monitoring` section controls W&B. Offline logs stay in the experiment directory; `enabled: false` disables monitoring. Gradient summaries are sampled, and full activations are never saved.


In [ ]:
monitor_config = config["monitoring"].copy()


## Optional: prepare the NPY cache

Set `prepare_npy_cache: true` in YAML when MAT features are added or replaced. The conversion remains fully expanded here; with the flag off, this cell does not scan or write the cache. It scans `data/` for `*_raw.mat`, excludes Archive/MergeOriginals, and mirrors relative folders under `python/dataset/`. Each recording produces `<user>_<activity>_<position>[_rNN]_time.npy` and `_stft.npy` as float32 arrays in the original `[N,270,T,3]` order. Labels stay in the filename; no normalization is applied here.

A small JSON completion record tracks source size/mtime and output sizes. Unchanged, complete pairs are skipped; missing outputs or changed source files are rebuilt. A partial conversion has no valid completion record and is retried. Source MAT files are retained. Memory holds one MAT variable at a time; no dataset-wide concatenation is performed. NPY files are uncompressed and require additional disk space.

The following cells use this cache directly through memory mapping. When no source files have changed, you can skip this conversion cell.


In [ ]:
converted = skipped = 0
sources = sorted(DATA_ROOT.rglob("*_raw.mat")) if prepare_npy_cache else []
for source in sources:
    relative = source.relative_to(DATA_ROOT)
    if {part.lower() for part in relative.parts} & {"archive", "mergeoriginals"}:
        continue
    folder = NPY_ROOT / relative.parent
    folder.mkdir(parents=True, exist_ok=True)
    stem = source.stem.removesuffix("_raw")
    outputs = {"segment_data": folder / f"{stem}_time.npy",
               "segment_data_stft": folder / f"{stem}_stft.npy"}
    marker = folder / f"{stem}_cache.json"
    stat = source.stat()
    signature = {"version": 1, "source": relative.as_posix(),
                 "source_bytes": stat.st_size, "source_mtime_ns": stat.st_mtime_ns}
    previous = {}
    if marker.exists():
        try:
            previous = json.loads(marker.read_text())
        except (ValueError, OSError):
            pass  # An incomplete completion record is rebuilt.
    if (previous.get("source_info") == signature
            and all(path.is_file() and path.stat().st_size == previous.get("output_bytes", {}).get(key)
                    for key, path in outputs.items())):
        skipped += 1
        continue

    shapes = {name: shape for name, shape, _ in whosmat(source)}
    a, b = shapes["segment_data"], shapes["segment_data_stft"]
    if (len(a) != 4 or len(b) != 4 or a[0] != b[0]
            or a[1::2] != (270, 3) or b[1::2] != (270, 3)):
        raise ValueError(f"Expected paired [N,270,T,3] arrays: {source}")
    marker.unlink(missing_ok=True)
    print(f"Converting: {relative}")
    for variable, destination in outputs.items():
        # Read one variable and cast one segment at a time to limit peak RAM.
        array = loadmat(source, variable_names=[variable])[variable]
        temporary = destination.with_suffix(".tmp.npy")
        mapped = np.lib.format.open_memmap(
            temporary, mode="w+", dtype=np.float32, shape=array.shape,
        )
        try:
            for i in range(len(array)):
                mapped[i] = array[i]
                if not np.isfinite(mapped[i]).all():
                    raise ValueError(f"Nonfinite float32 features: {source}, {variable}, segment {i}")
            mapped.flush()
        finally:
            del mapped, array
        temporary.replace(destination)
    current = source.stat()
    if (current.st_size, current.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
        raise RuntimeError(f"Source changed during conversion; rerun this cell: {source}")
    completed = {"source_info": signature,
                 "output_bytes": {key: path.stat().st_size for key, path in outputs.items()}}
    temporary_marker = marker.with_suffix(".tmp.json")
    temporary_marker.write_text(json.dumps(completed, indent=2))
    temporary_marker.replace(marker)
    converted += 1

if prepare_npy_cache:
    print(f"NPY cache: {converted} converted, {skipped} skipped. Output: {NPY_ROOT}")
else:
    print("Using the existing NPY cache; conversion is disabled.")


## Find NPY pairs and read labels

Scan `_time.npy` filenames and pair each with `_stft.npy`. Completion records and NPY headers verify paired shapes without loading the full arrays. `metadata` retains all filename fields. Archive and MergeOriginals are excluded.

In [ ]:
parts = []
for token in re.split(r"(\{[^{}]+\}|\[|\])", data_config["filename_template"]):
    if token.startswith("{"):
        parts.append(fr"(?P<{token[1:-1]}>[^_/\\]+?)")
    elif token == "[":
        parts.append("(?:")
    elif token == "]":
        parts.append(")?")
    else:
        parts.append(re.escape(token))
pattern = re.compile("".join(parts))

files, stft_files, metadata, counts, shapes = [], [], [], [], []
paths = set()
for root in data_config["roots"]:
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(root)
    for path in root.rglob("*_time.npy"):
        if {p.lower() for p in path.relative_to(root).parts} & {"archive", "mergeoriginals"}:
            continue
        paths.add(path.resolve())  # Parent/child or repeated roots must not duplicate samples.

for path in sorted(paths):
    match = pattern.fullmatch(path.name)
    if match is None:
        raise ValueError(f"Filename does not match template: {path}")
    fields = {k: v for k, v in match.groupdict().items() if v is not None}
    stem = path.name.removesuffix("_time.npy")
    stft_path = path.with_name(stem + "_stft.npy")
    marker = path.with_name(stem + "_cache.json")
    if not stft_path.is_file() or not marker.is_file():
        raise ValueError(f"Incomplete NPY pair; rerun the conversion cell: {path}")
    completed = json.loads(marker.read_text())
    if (path.stat().st_size != completed["output_bytes"]["segment_data"]
            or stft_path.stat().st_size != completed["output_bytes"]["segment_data_stft"]):
        raise ValueError(f"Incomplete NPY output; rerun the conversion cell: {path}")
    time = np.load(path, mmap_mode="r")
    stft = np.load(stft_path, mmap_mode="r")
    time_shape, stft_shape = time.shape, stft.shape
    if time.dtype != np.float32 or stft.dtype != np.float32:
        raise ValueError(f"Expected float32 cache: {path}")
    del time, stft
    if (len(time_shape) != 4 or len(stft_shape) != 4
            or time_shape[0] != stft_shape[0]
            or time_shape[1::2] != (270, 3) or stft_shape[1::2] != (270, 3)):
        raise ValueError(f"Expected paired [N,270,T,3] arrays: {path}")
    if time_shape[0] == 0:
        continue
    files.append(path)
    stft_files.append(stft_path)
    metadata.append(fields)
    counts.append(time_shape[0])
    shapes.append((time_shape[1:], stft_shape[1:]))

if not files or len(set(shapes)) != 1:
    raise ValueError("Select a nonempty dataset with consistent feature shapes")
labels = np.array([m[data_config["task"]] for m in metadata])
counts = np.array(counts)
time_shape, stft_shape = shapes[0]
size_gib = counts.sum() * (np.prod(time_shape) + np.prod(stft_shape)) * 4 / 1024**3
print(f"{len(files)} files, {counts.sum()} segments, {len(set(labels))} labels")
print(f"NPY arrays on disk: {size_gib:.2f} GiB; accessed through memory mapping")


## Split source and target data

`domain_holdout` reserves every segment of recordings matching `split.test_domains` for test. `split.validation_domains` optionally isolates whole validation domains, excluded from both training and calibration. Otherwise validation is sampled from source domains. Remaining source recordings are expanded into `(file_index, segment_index)` pairs and partitioned within each domain/class. Fractions apply within each group, rounded down with at least one segment per requested held-out split; groups too small to retain a training segment are rejected. Signal arrays remain on disk.

Source subsets may share recordings, but never the same segment. The target domain never enters any source subset. `test_fraction` is unused in this mode. The selected YAML determines the held-out users.

`random` retains the original four-way recording split. Training defines the label vocabulary; unseen held-out labels are rejected. Inspect the printed domain/class counts. File counts in source subsets need not sum to the dataset file count when splitting segments.

Weighted CP estimates the target/source ratio from unlabeled test inputs. No extra domains are reserved. Target labels never enter ratio fitting.


In [ ]:
from collections import Counter
from step00_config.schema import WeightingConfig

weighting_settings = WeightingConfig(**globals().get("config", {}).get("weighting", {}))
from step01_preprocessing.splitting import split_samples as make_split

split_samples = make_split(labels, counts, metadata, data_config, split_config, split_seed)
mode = split_config["mode"]
# Preserve recording order for both segment and recording-level splits.
split_indices = {name: samples[np.sort(np.unique(samples[:, 0], return_index=True)[1]), 0]
                 for name, samples in split_samples.items()}
domains = np.array([m[data_config["domain_key"]] for m in metadata])

# Class vocabulary comes from training, never from held-out labels.
class_names = sorted(set(labels[split_indices["train"]].tolist()))
label_map = {name: i for i, name in enumerate(class_names)}
for name, idx in split_indices.items():
    unseen = set(labels[idx]) - set(class_names)
    if unseen:
        raise ValueError(f"{name} contains labels absent from training: {sorted(unseen)}")
targets = np.array([label_map[label] for label in labels])
split_unit = "segment" if mode == "domain_holdout" else "recording"
print("Source split unit:", split_unit)
print("Label map:", label_map)
split_summary = {}
for name, idx in split_indices.items():
    summary = {"files": len(idx), "segments": len(split_samples[name]),
               "class_files": dict(sorted(Counter(labels[idx].tolist()).items())),
               "class_segments": dict(sorted(Counter(labels[split_samples[name][:, 0]].tolist()).items()))}
    if mode == "domain_holdout":
        summary["domain_files"] = dict(sorted(Counter(domains[idx].tolist()).items()))
        summary["domain_segments"] = dict(sorted(Counter(domains[split_samples[name][:, 0]].tolist()).items()))
    split_summary[name] = summary
    print(name, summary)


## Create memory-mapped DataLoaders

The shared Dataset maps NPY files read-only and copies only the requested segment. No full-dataset tensor or concatenation is created. The most recently accessed recording pair remains mapped; changing recordings releases that pair. OS file caching is reclaimable and is separate from allocating the complete dataset in Python.

Within each segment, the three time-domain feature groups are standardized separately across the 270 rows and time axis. The standard-deviation floor uses 270*T elements per group. STFT magnitude stays on its stored scale; no log transform or standardization is applied to STFT. Batches start with `(time, stft, labels)`. Training and validation also contain training-vocabulary domain IDs (unseen domains use -1); validation additionally contains separate grouping IDs so unseen users retain independent CE/accuracy means. The first three entries have shapes `[B,3,270,T]`, `[B,3,270,F_times_frames]`, `[B]`. With `sampling.enabled`, training uses full class/domain-balanced batches and its configured batch size; small groups may be oversampled. Otherwise training shuffles using `training.batch_size`, retaining singleton tail batches with batch-independent normalization. Other splits retain their original order and all samples. Start with `num_workers=0` to avoid multiprocessing/prefetch overhead on the local machine.

In [ ]:
# Head IDs come from training only; separate group IDs identify unseen validation users.
key = data_config["domain_key"]
training_records = split_indices["train"]
domain_names = sorted({metadata[i][key] for i in training_records})
domain_map = {name: i for i, name in enumerate(domain_names)}
domain_targets = np.array([domain_map.get(m[key], -1) for m in metadata], dtype=np.int64)
validation_domains = sorted({metadata[i][key] for i in split_indices["validation"]})
validation_map = {name: i for i, name in enumerate(validation_domains)}
validation_groups = np.array([validation_map.get(m[key], -1) for m in metadata], dtype=np.int64)
print("Training domain map:", domain_map)
print("Validation groups (equal weight):", validation_domains)
if adversarial_config.enabled or contrastive_config.enabled or sampling_config.enabled or encoder_background_config.enabled:
    if key == data_config["task"]:
        raise ValueError("Training domain must differ from the prediction task")
    if len(domain_names) < 2:
        raise ValueError("Domain-aware training requires at least two training domains")
    if contrastive_config.enabled:
        train_files = split_samples["train"][:, 0]
        paired_classes = [label for label in np.unique(targets[train_files])
                          if len(np.unique(domain_targets[train_files][targets[train_files] == label])) >= 2]
        if not paired_classes:
            raise ValueError("Contrastive learning requires a task class in at least two training domains")
        print(f"Classes with cross-domain positives: {len(paired_classes)}/{len(class_names)}")


from step01_preprocessing.dataset import NpySegments

from step03_training.sampling import ClassDomainSampler

train_sampler = None
if sampling_config.enabled:
    train_files = split_samples["train"][:, 0]
    train_sampler = ClassDomainSampler(targets[train_files], domain_targets[train_files],
                                       train_files, sampling_config, training_seed)
    print(f"Balanced training batch size: {train_sampler.batch_size}; "
          f"sample draws per epoch: {len(train_sampler)}")

datasets, loaders = {}, {}
for name, samples in split_samples.items():
    datasets[name] = NpySegments(files, stft_files, targets, samples,
                                 domain_targets if name in ("train", "validation") else None,
                                 group_labels=validation_groups if name == "validation" else None)
    sampler = train_sampler if name == "train" else None
    batch_size = sampler.batch_size if sampler is not None else training_config.batch_size
    loaders[name] = DataLoader(
        datasets[name], batch_size=batch_size, sampler=sampler,
        shuffle=(name == "train" and sampler is None), num_workers=0,
        drop_last=False,
        generator=torch.Generator().manual_seed(training_seed),
    )
train_loader = loaders["train"]
val_loader = loaders["validation"]
print({name: len(dataset) for name, dataset in datasets.items()})

## Inspect a batch

Use validation here so inspecting shapes does not advance the training shuffle generator.

In [ ]:
time_batch, stft_batch, label_batch = next(iter(val_loader))[:3]
print(time_batch.shape, stft_batch.shape, label_batch.shape)
print(time_batch.dtype, stft_batch.dtype, label_batch.dtype)

## Build and inspect the selected model

`model.kind` selects the task backbone. `encoder_background.enabled` adds a shared background
encoder, optional domain/activity heads, and a decoder. Classification still uses only
the task embedding. Source users share the same background encoder; unseen target inputs
use that saved encoder without adaptation. The master switch defaults to false.

Background activity adversarial conditioning is controlled independently by `encoder_background.adversarial.conditional`. When enabled, the head receives the true training-domain one-hot; task prediction and CP do not require this condition. Conditional activity diagnostics skip unknown domains and report the evaluated sample count.

The activity adversary width is controlled independently by `encoder_background.adversarial.hidden_dim`; baseline uses 128 for both adversaries.

Both adversarial heads share `training.optimizer` and `training.scheduler` with the encoders and decoder. `data.domain_key` is shared by both branches, CL, sampling and weighting. `decoder` controls reconstruction and shares the main optimizer. Each encoder difference loss detaches the opposite embedding; both are inactive without background.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if (augmentation_config.enabled and augmentation_config.stft_mask["enabled"]
        and augmentation_config.stft_mask["frequency_bins"] != model_config.stft_frequency_bins):
    raise ValueError("Model and STFT masking must use the same frequency_bins")
model = build_model(model_config, time_shape, stft_shape, len(class_names), training_seed,
                      domain_classes=len(domain_names) if adversarial_config.enabled else 0,
                      domain_hidden_dim=adversarial_config.hidden_dim,
                      domain_conditional=adversarial_config.conditional,
                      encoder_background=encoder_background_config, background_domains=len(domain_names), decoder=decoder_config).to(device)
print(model)
print(f"Device: {device}; parameters: {sum(p.numel() for p in model.parameters()):,}")
# Inference smoke check; disables Dropout and does not consume training shuffle.
model.eval()
with torch.inference_mode():
    logits, embedding = model(time_batch[:inference_batch_size].to(device),
                              stft_batch[:inference_batch_size].to(device), return_embedding=True)
print("Logits:", tuple(logits.shape), "Embedding:", tuple(embedding.shape))

## Train and validate

Run experiment setup, optimizer setup, the training loop, and best-model restoration in order. Only the loop cell starts training. Domain-mean validation CE selects weights; task patience and active DA/CL stability control early stopping. Calibration/test stay outside training.

Optional domain-adversarial training adds a small classifier on the shared embedding. Its domain vocabulary comes only from training records. Gradient reversal suppresses domain information in the encoder while the domain classifier minimizes cross-entropy. Its strength ramps linearly over `encoder_task.adversarial.warmup_epochs`; this is source-domain generalization, with no target-domain training. `train_loss` remains task cross-entropy; domain loss, accuracy and reversal weight are logged separately. All enabled modules share one optimizer and scheduler: each batch has one backward pass and one optimizer step. Gradient reversal scales only encoder gradients; adversarial heads minimize their CE even when the reversal weight is zero. Validation domain metrics use clean inputs in eval mode; unknown validation domains are excluded and `validation_domain_samples` reports the count. Joint stopping monitors training DA loss/accuracy and CL loss, so isolated validation domains can also be used. Low domain accuracy alone does not establish invariance.

`encoder_task.adversarial.conditional` supplies the true task-label one-hot vector only to the domain head; the encoder and task head never receive labels as inputs. `encoder_task.contrastive.enabled` adds a weighted loss on normalized embeddings, pulling same-class/different-domain pairs together and separating different classes. Same-class/same-domain pairs are excluded. It has its own switch, weight, and temperature; GRL still applies the adversarial multiplier exactly once. The contrastive loss and valid-anchor fraction are logged separately.

`sampling.enabled` supplies cross-domain pairs using class/domain-balanced training batches. All domain-aware components share `data.domain_key`. Classes missing domain combinations are retained; only anchors with cross-domain positives enter the contrastive mean. Embedding export always traverses the original split, not the balanced sampler.

Each experiment keeps one best checkpoint, a split/configuration manifest and training history. History is overwritten after each completed epoch, so interruption preserves previous epochs without accumulating epoch files. Starting a fresh reproducible run requires rerunning the DataLoader and model cells as well as training setup: the loader has its own shuffle generator.

W&B uses `monitor_config` from the settings section. Batch metrics use `train/step`; epoch metrics use `epoch`. Sampled gradients under `gradients/activations/` are dL/d(layer output), and those under `gradients/parameters/` are dL/d(parameter). Histograms contain weights and gradients, not full activation tensors. Validation does not collect diagnostics, and context cleanup removes hooks on completion or interruption.

For live dashboards, run `wandb login` in the wifi-cp environment and select online mode. Offline logs can be uploaded later with `wandb sync <offline-run-directory>`. Neither mode uploads raw CSI or checkpoints through this monitor.

W&B initialization errors stop before training. Once initialized, logging failures warn once and disable monitoring; training and local saves continue. Model errors and interrupts still propagate.


In [ ]:
from datetime import datetime, timezone

run_dir = (Path(experiment_dir) if experiment_dir else
           PYTHON_ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
run_dir.mkdir(parents=True, exist_ok=experiment_dir is not None)
if any((run_dir / name).exists() for name in ("experiment.json", "best.pt", "last.pt")):
    raise FileExistsError(f"Existing experiment; choose a new experiment_dir: {run_dir}")
encoder_task_config = EncoderTaskConfig(task=encoder_task_config.task, difference=encoder_task_config.difference,
    contrastive=asdict(contrastive_config), adversarial=asdict(adversarial_config))
experiment = {
    "split_seed": split_seed, "training_seed": training_seed,
    "task": data_config["task"], "class_names": class_names,
    "split": split_config, "split_summary": split_summary,
    "weighting": asdict(weighting_settings),
    "label_map": label_map, "roots": [str(p) for p in data_config["roots"]],
    "time_shape": list(time_shape), "stft_shape": list(stft_shape),
    "data": {"task": data_config["task"], "domain_key": data_config["domain_key"]},
    "decoder": asdict(decoder_config),
    "model": asdict(model_config), "encoder_background": asdict(encoder_background_config),
    "training": {**asdict(training_config), "batch_size": train_loader.batch_size},
    "augmentation": asdict(augmentation_config),
    "encoder_task": asdict(encoder_task_config), "sampling": asdict(sampling_config),
    "domain_names": domain_names,
    "monitoring": monitor_config,
    "preprocessing": {"time": "per_sample_per_group_standardize", "stft": "raw_magnitude"},
    "torch_version": str(torch.__version__),
    "split_unit": split_unit,
    "recordings": [{"time": str(path.relative_to(NPY_ROOT) if path.is_relative_to(NPY_ROOT) else path),
                    "stft": str(stft_files[i].relative_to(NPY_ROOT) if stft_files[i].is_relative_to(NPY_ROOT) else stft_files[i]),
                    "label": int(targets[i]), "segments": int(counts[i]), "metadata": metadata[i]}
                   for i, path in enumerate(files)],
    "split_index_columns": ["recording_index", "segment_index"],
    "splits": {name: samples.tolist() for name, samples in split_samples.items()},
}
(run_dir / "experiment.json").write_text(json.dumps(experiment, indent=2))
# Persist interactive YAML settings even if training or CP stops early.
if not (run_dir / "config.yaml").exists():
    import yaml
    (run_dir / "config.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
# Keep the full split manifest local; send experiment settings and split sizes to W&B.
monitor_experiment = {
    key: experiment[key] for key in ("split_seed", "training_seed", "augmentation",
                                    "encoder_task", "sampling", "domain_names", "encoder_background",
                                    "data", "decoder", "task", "class_names", "split", "split_unit", "split_summary",
                                    "model", "training", "time_shape", "stft_shape", "preprocessing", "torch_version")
}
monitor_experiment["split_segments"] = {name: len(ds) for name, ds in datasets.items()}


## Prepare optimizer, scheduler and augmentation

This cell initializes training state without processing batches.

In [ ]:
import random
from step03_training.schedulers import build_scheduler
from step03_training.augmentation import BatchAugmenter
from step03_training.contrastive import cross_domain_contrastive
from step00_config.schema import SchedulerConfig
scheduler_config = SchedulerConfig(**training_config.scheduler)

from step00_config.schema import DecoderConfig, EncoderBackgroundConfig, EncoderTaskConfig
from step03_training.objectives import joint_objective
from step03_training.validation import DomainMetrics, background_validation
from step03_training.early_stopping import JointEarlyStopping
from step03_training.checkpoints import save_checkpoint
encoder_background_config = globals().get("encoder_background_config", EncoderBackgroundConfig())
decoder_config = globals().get("decoder_config", DecoderConfig())
# Notebook helpers can supply task CE settings while retaining typed CL/DA configs.
encoder_task_config = EncoderTaskConfig(
    task=globals().get("encoder_task_config", EncoderTaskConfig()).task,
    difference=globals().get("encoder_task_config", EncoderTaskConfig()).difference,
    contrastive=asdict(contrastive_config), adversarial=asdict(adversarial_config))
task_ce_weight = encoder_task_config.task["weight"] if encoder_task_config.task["enabled"] else 0.0
# Check settings before updating any weights.
if training_config.learning_rate <= 0 or training_config.weight_decay < 0:
    raise ValueError("Invalid learning rate or weight decay")
if not len(train_loader) or not len(val_loader):
    raise ValueError("Training and validation must both be nonempty")
early_stopping = JointEarlyStopping(training_config, adversarial_config, contrastive_config, encoder_background_config)
if training_config.epochs <= early_stopping.warmup:
    raise ValueError("training.epochs must exceed active DA warmup_epochs to select a post-warmup best checkpoint")
checkpoint_path = run_dir / "best.pt"

random.seed(training_seed)
np.random.seed(training_seed)
torch.manual_seed(training_seed)
model.to(device)
optimizers = {"adamw": torch.optim.AdamW, "adam": torch.optim.Adam, "sgd": torch.optim.SGD,
              "rmsprop": torch.optim.RMSprop}
optimizer = optimizers[training_config.optimizer](
    model.parameters(), lr=training_config.learning_rate,
    weight_decay=training_config.weight_decay,
    **training_config.optimizer_kwargs,
)
scheduler = build_scheduler(optimizer, scheduler_config, training_config.epochs)
augmenter = BatchAugmenter(augmentation_config, training_seed, device)
criterion = torch.nn.CrossEntropyLoss()
history = []
best_loss = float("inf")
best_epoch = global_step = 0

checkpoint_metadata = {
    "model_config": getattr(model, "architecture_config", None),
    "encoder_background": asdict(encoder_background_config), "decoder": asdict(decoder_config),
    "training_config": {**asdict(training_config), "batch_size": train_loader.batch_size},
    "split_seed": split_seed, "training_seed": training_seed,
    "augmentation": asdict(augmentation_config), "encoder_task": asdict(encoder_task_config),
    "sampling": asdict(sampling_config), "domain_names": domain_names,
}
print("Early stopping monitors: domain-mean validation CE +", early_stopping.monitors or "no auxiliary objectives")

print(f"Best checkpoint selection starts at epoch {early_stopping.warmup + 1}")


## Train and validate

The complete batch and epoch loops are below. Run this cell to start training. W&B hooks are removed when the context exits, including on interruption.

In [ ]:
# Rerun preparation with a fresh run directory before starting another experiment.
if checkpoint_path.exists() or (run_dir / "last.pt").exists():
    raise FileExistsError(f"Choose a new run directory: {run_dir}")

with TrainingMonitor(model, monitor_config, run_dir=run_dir,
                     experiment=monitor_experiment) as monitor:
    for epoch in range(1, training_config.epochs + 1):
        # 1. Train: forward -> loss -> backward -> parameter update.
        model.train()
        epoch_learning_rate = optimizer.param_groups[0]["lr"]
        train_loss_sum = train_objective_sum = train_correct = train_count = 0
        domain_loss_sum = domain_correct = weight_sum = 0
        contrastive_loss_sum = contrastive_count = 0
        background_metric_sums, background_metric_counts = {}, {}
        optimizer_updated = False
        for batch in train_loader:
            time, stft, labels = batch[:3]
            time, stft, labels = (x.to(device) for x in (time, stft, labels))
            time, stft = augmenter(time, stft)
            global_step += 1
            monitor.begin_step(global_step)
            optimizer.zero_grad(set_to_none=True)
            if adversarial_config.enabled or contrastive_config.enabled or encoder_background_config.enabled:
                logits, embedding = model(time, stft, return_embedding=True)
            else:
                logits = model(time, stft)
            task_loss = criterion(logits, labels)
            # Do not attach disabled CE: even zero gradients would trigger optimizer decay/momentum.
            loss = task_ce_weight * task_loss if task_ce_weight else logits.new_zeros(())
            batch_metrics = {}
            if contrastive_config.enabled:
                contrastive_loss, anchors = cross_domain_contrastive(
                    embedding, labels, batch[3].to(device), contrastive_config.temperature)
                if contrastive_config.weight:
                    loss = loss + contrastive_config.weight * contrastive_loss
                contrastive_loss_sum += contrastive_loss.item() * anchors
                contrastive_count += anchors
                batch_metrics.update({"train/contrastive_loss": contrastive_loss.item(),
                                      "train/contrastive_anchor_fraction": anchors / len(labels)})
            if adversarial_config.enabled:
                # Only the encoder gradient is reversed; the domain head minimizes CE.
                ramp_steps = adversarial_config.warmup_epochs * len(train_loader)
                weight = adversarial_config.max_weight * (min(1.0, (global_step - 1) / ramp_steps)
                                                         if ramp_steps else 1.0)
                domain_labels = batch[3].to(device)
                domain_logits = model.domain_classifier(embedding, weight, labels=labels)
                domain_loss = criterion(domain_logits, domain_labels)
                loss = loss + domain_loss
                domain_loss_sum += domain_loss.item() * len(labels)
                domain_correct += (domain_logits.argmax(1) == domain_labels).sum().item()
                weight_sum += weight * len(labels)
                batch_metrics.update({"train/domain_loss": domain_loss.item(),
                                      "train/adversarial_weight": weight})
            if encoder_background_config.enabled:
                background_loss, background_metrics = joint_objective(
                    model, embedding, time, stft, labels, batch[3].to(device),
                    encoder_background_config, global_step, len(train_loader),
                    task_difference=encoder_task_config.difference, decoder=decoder_config)
                loss = loss + background_loss
                for key, metric in background_metrics.items():
                    # Match task CL: loss is averaged over valid anchors, not all batch samples.
                    count = (background_metrics["background_anchor_fraction"] * len(labels)
                             if key == "background_contrastive_loss" else len(labels))
                    background_metric_sums[key] = background_metric_sums.get(key, 0.) + metric * count
                    background_metric_counts[key] = background_metric_counts.get(key, 0.) + count
                    batch_metrics["train/" + key] = metric
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss")
            if loss.requires_grad:
                loss.backward()
                monitor.after_backward(model)
                optimizer.step()
                optimizer_updated = True
            monitor.end_step({**batch_metrics, "train/batch_loss": task_loss.item(),
                              "train/objective_loss": loss.item(), "train/task_ce_weight": task_ce_weight,
                              "train/learning_rate": optimizer.param_groups[0]["lr"]})
            train_objective_sum += loss.item() * len(labels)
            train_loss_sum += task_loss.item() * len(labels)
            train_correct += (logits.argmax(1) == labels).sum().item()
            train_count += len(labels)

        # 2. Validate: no parameter updates or Dropout.
        model.eval()
        validation_metrics = DomainMetrics()
        val_domain_loss_sum = val_domain_correct = val_domain_count = 0
        background_validation_sums = {key: [0., 0, 0] for key in ("domain", "activity")}
        with torch.no_grad():
            for batch in val_loader:
                time, stft, labels = batch[:3]
                time, stft, labels = (x.to(device) for x in (time, stft, labels))
                if adversarial_config.enabled:
                    logits, embedding = model(time, stft, return_embedding=True)
                    domain_labels = batch[3].to(device)
                    known = domain_labels >= 0
                    if known.any():
                        domain_logits = model.domain_classifier(embedding[known], labels=labels[known])
                        domain_loss = criterion(domain_logits, domain_labels[known])
                        if not torch.isfinite(domain_loss):
                            raise ValueError("Nonfinite validation domain loss")
                        count = known.sum().item()
                        val_domain_loss_sum += domain_loss.item() * count
                        val_domain_correct += (domain_logits.argmax(1) == domain_labels[known]).sum().item()
                        val_domain_count += count
                else:
                    logits = model(time, stft)
                if encoder_background_config.enabled:
                    for key, (ce, correct, count) in background_validation(
                            model, time, stft, labels, batch[3].to(device)).items():
                        totals = background_validation_sums[key]
                        totals[0] += ce * count
                        totals[1] += correct
                        totals[2] += count
                groups = batch[4] if len(batch) > 4 else (batch[3] if len(batch) > 3 else torch.zeros_like(labels))
                validation_metrics.update(logits, labels, groups, criterion)

        # 3. Training is sample-weighted; task validation gives each domain equal weight.
        row = {
            "epoch": epoch,
            "learning_rate": epoch_learning_rate,
            "train_loss": train_loss_sum / train_count,
            "train_objective_loss": train_objective_sum / train_count,
            "task_ce_weight": task_ce_weight,
            "train_accuracy": train_correct / train_count,
            "train_samples": train_count,
            **validation_metrics.compute(),
        }
        if adversarial_config.enabled:
            row.update(train_domain_loss=domain_loss_sum / train_count,
                       train_domain_accuracy=domain_correct / train_count,
                       adversarial_weight=weight_sum / train_count,
                       validation_domain_loss=val_domain_loss_sum / val_domain_count if val_domain_count else None,
                       validation_domain_accuracy=val_domain_correct / val_domain_count if val_domain_count else None,
                       validation_domain_samples=val_domain_count)
        if contrastive_config.enabled:
            row.update(train_contrastive_loss=contrastive_loss_sum / contrastive_count if contrastive_count else None,
                       train_contrastive_anchor_fraction=contrastive_count / train_count)
        row.update({"train_" + key: total / background_metric_counts[key] if background_metric_counts[key] else None
                    for key, total in background_metric_sums.items()})
        if encoder_background_config.enabled:
            for key, (loss_sum, correct, count) in background_validation_sums.items():
                head_enabled = (encoder_background_config.domain["enabled"] if key == "domain"
                                else encoder_background_config.adversarial["enabled"])
                if head_enabled:
                    row.update({f"validation_background_{key}_loss": loss_sum / count if count else None,
                                f"validation_background_{key}_accuracy": correct / count if count else None,
                                f"validation_background_{key}_samples": count})
        history.append(row)
        value = row["validation_loss"]

        # Schedule the next epoch; history records the LR used in this epoch.
        if scheduler is not None and optimizer_updated:
            if scheduler_config.name == "plateau":
                scheduler.step(value)
            else:
                scheduler.step()

        # 4. Select best only after active DA warmup; always preserve the latest epoch.
        save_checkpoint(run_dir / "last.pt", model, epoch, value, checkpoint_metadata)
        if epoch > early_stopping.warmup and value < best_loss:
            best_loss, best_epoch = value, epoch
            save_checkpoint(checkpoint_path, model, epoch, value, checkpoint_metadata)

        # 5. Task patience and active DA/CL stability must both be satisfied after warmup.
        should_stop = early_stopping.update(row)
        row.update(early_stopping_wait=early_stopping.wait,
                   auxiliary_stable_epochs=early_stopping.stable_epochs)
        # Keep completed epochs on disk even if a later epoch is interrupted.
        result = {"history": history, "best_epoch": best_epoch or None,
                  "best_validation_loss": best_loss if best_epoch else None,
                  "validation_metric": "domain_mean_ce",
                  "early_stopping_monitors": early_stopping.monitors,
                  "epochs_run": len(history), "stopped_early": should_stop and epoch < training_config.epochs}
        temporary = run_dir / "history.json.tmp"
        temporary.write_text(json.dumps(result, indent=2))
        temporary.replace(run_dir / "history.json")
        monitor.log_epoch(epoch, {k: v for k, v in row.items() if k != "epoch"})

        print(f"Epoch {epoch:03d} | train loss {row['train_loss']:.4f}, acc {row['train_accuracy']:.3f}"
              f" | val domain-mean loss {value:.4f}, acc {row['validation_accuracy']:.3f}"
              f" | lr {epoch_learning_rate:.3g}", flush=True)
        if encoder_background_config.enabled:
            for head, title in (("domain", "background domain"), ("activity", "background activity adversary")):
                key = f"validation_background_{head}"
                if row.get(f"{key}_samples", 0):
                    print(f"  {title}: val loss {row[f'{key}_loss']:.4f}, "
                          f"acc {row[f'{key}_accuracy']:.3f}, n={row[f'{key}_samples']}", flush=True)
        if early_stopping.patience is not None:
            print(f"  stopping: task wait {early_stopping.wait}/{early_stopping.patience}, "
                  f"auxiliary stable {early_stopping.stable_epochs}/{early_stopping.patience}, "
                  f"warmup {early_stopping.warmup}", flush=True)
        if should_stop:
            break

    monitor.log_result(result)

## Restore the best model

History and `last.pt` are updated after each completed epoch; `best.pt` saves every new absolute minimum of domain-mean validation CE after active DA warmup (epoch > warmup). Early-stopping task patience also compares only post-warmup CE. With no active DA warmup, selection starts at epoch 1. Before the first eligible epoch, history records null best metrics and only `last.pt` is saved. Run this cell to restore `best.pt` before exporting features. To compare the final epoch, use evaluate with `checkpoint_name="last.pt"` and `recompute=True`.


In [ ]:
from step03_training.cleanup import release_training_state
release_training_state(globals(), model)

# Restore the best epoch, which may differ from the final epoch.
if not checkpoint_path.is_file():
    raise FileNotFoundError("No post-warmup best checkpoint; training may have stopped during warmup. "
                            "Use evaluate with checkpoint_name=\"last.pt\" to inspect the last completed epoch.")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
del checkpoint
model.eval()
print("Best epoch:", best_epoch, "Validation loss:", best_loss)
print("Saved to:", run_dir)


In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, metric in zip(axes, ("loss", "accuracy")):
    for split in ("train", "validation"):
        ax.plot(epochs, [r[f"{split}_{metric}"] for r in history], label=split)
    if result.get("best_epoch") is not None:
        ax.axvline(result["best_epoch"], color="gray", linestyle="--", label="best epoch")
    ax.set(xlabel="Epoch", ylabel=metric.capitalize())
    ax.legend()
axes[2].plot(epochs, [r["learning_rate"] for r in history])
axes[2].set(xlabel="Epoch", ylabel="Learning rate")
fig.tight_layout()
fig.savefig(run_dir / "training.png", dpi=150)
plt.show()
plt.close(fig)

## Export embeddings

Export each saved split once, in original row order, using inference.batch_size. All inputs are clean; training augmentation is not applied. The cached task/background representations are shared by score fitting, target weight estimation and prediction. Target labels are used only for evaluation metrics.

In [ ]:
from evaluation_pipeline import ConformalEvaluation

## Fit the score model

Choose a score model and its settings here. The configured `score.kind` selects the scorer. KDE uses per-class densities on the embedding. SVM/HBGB are alternatives using probability-margin scores. All scores have shape [samples, classes]; larger values mean less conformity. Point predictions use the minimum score, not the maximum calibrated p-value.

`score.kind: softmax` directly uses `1 - p_y` from the frozen NN probabilities and skips intermediate model fitting. For pooled ordinary split CP, set `weighted_cp: false`; calibration pools all true-label scores for any score choice.

Fit KDE/SVM/HBGB only on training embeddings. Use validation accuracy to inspect/choose settings, then freeze them before running Step 06. No additional embedding normalization is applied. KDE bandwidth is in embedding units. The predeclared alpha grid below is for reporting, not selecting a threshold from test performance.
SVM probabilities use sigmoid calibration with five seeded stratified folds inside the training split (`CalibratedClassifierCV`, `ensemble=False`). Each training class needs at least five samples. This probability calibration is separate from Step 06 rank calibration.


In [ ]:
# Recreate the session after changing evaluation settings.
evaluation = None
evaluation = ConformalEvaluation(model, datasets, experiment, config, checkpoint_path,
                                  settings_source="run configuration", aps_seed=config.get("aps_seed", 42))

In [ ]:
from step03_training.cleanup import release_training_state
release_training_state(globals(), model)
evaluation.export()

In [ ]:
evaluation.fit_score()

## Calibrate the score model

Run this after model/score settings have been selected with validation. Calibration uses only the score of each sample's true class. Pooled calibration uses all true-label scores in one reference distribution. Ties are included, with the finite-sample +1 correction.

Each invocation creates a separate conformal result directory. Do not tune the model, scoring settings or alpha using calibration/test results. These are segment-level empirical evaluations: segments from one recording can be dependent, so file-disjoint splitting alone does not establish exchangeability of individual segments or a distribution-free coverage guarantee.

Randomized APS is calibrated alongside the configured score: unweighted APS always runs, and weighted APS uses the same calibration/test weights when weighted_cp is enabled. Calibration and test use independent random streams; draws stay fixed across the alpha sweep. No forced nonempty sets are added.

In [ ]:
evaluation.calibrate()

## Predict sets on test data

For candidate class c, p_c = (1 + number of reference scores >= candidate score) / (reference count + 1). Include c exactly when p_c > alpha. These p-values are not class probabilities and need not sum to one. Test labels are used only for subsequent evaluation, not scoring or set construction.
With `weighted_cp: true`, each count is replaced by a sum of weights, and the +1 is replaced by the candidate test point's own weight. Score ties remain included. Weights estimated from reused unlabeled target inputs do not establish an exact target coverage guarantee.


In [ ]:
evaluation.predict()

## Evaluate and save results

Coverage includes empty prediction sets. Report the scorer's point-classification accuracy separately from CNN accuracy. Per-class results and the fixed alpha sweep are descriptive evaluations; do not use test outcomes to select a configuration. Every test segment receives equal weight, so recordings with more segments contribute more to these metrics.

The common coverage/set-size plot compares configured CP, APS unweighted, and APS weighted when enabled, with nominal coverage and NN accuracy. All results and APS draws are saved together in the new conformal directory.

In [ ]:
cp_dir = None
cp_dir = evaluation.save_results()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

evaluation.require_saved_results()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, rows in evaluation.method_results.items():
    levels = [row["alpha"] for row in rows]
    axes[0].plot(levels, [row["coverage"] for row in rows], "o-", label=name)
    axes[1].plot(levels, [row["mean_set_size"] for row in rows], "o-", label=name)
axes[0].plot(levels, 1 - np.asarray(levels), "--", color="gray", label="Nominal 1-alpha")
axes[0].axhline(evaluation.nn_accuracy, color="tab:red", linestyle="-.", label=f"NN accuracy = {evaluation.nn_accuracy:.3f}")
axes[0].set(xlabel="Alpha", ylabel="Coverage / accuracy", ylim=(0, 1.02))
axes[1].set(xlabel="Alpha", ylabel="Mean prediction-set size", ylim=(0, len(evaluation.class_names)))
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle("CP and randomized APS comparison")
fig.tight_layout()
fig.savefig(evaluation.cp_dir / "coverage_set_size.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
from step07_evaluation.monitoring import log_evaluation
evaluation.require_saved_results()
log_evaluation(evaluation, monitor_config)

## Compare embedding projections

1. Load and sample saved embeddings, keeping labels and recording/segment identifiers aligned.
2. Choose methods and parameters, compute coordinates, and save NPZ files.
3. Adjust plot styling and save PNG files using the cached coordinates. Rerun only this cell to change colors, markers, legends or layout.

The default methods are t-SNE and metric MDS; Isomap and Spectral Embedding are optional entries in the computation cell. Reduce `viz_max_per_split` when pairwise-distance methods are expensive. Graph disconnection warnings matter; projections alone do not establish domain invariance.

In [ ]:
from step07_evaluation.projections import load_projection_data

viz_dir = cp_dir  # Or a saved conformal-* directory.
viz_seed = 42
viz_max_per_split = 1000  # None keeps every segment.
# Invalidate old coordinates before changing the selected data.
viz_data = projections = None
viz_data = load_projection_data(viz_dir, viz_seed, viz_max_per_split)

In [ ]:
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding, MDS
import json
import numpy as np

# Changing these parameters requires rerunning this cell; style changes do not.
projections = None
if viz_data is not None:
    embeddings = viz_data["embeddings"]
    map_neighbors = min(30, len(embeddings) - 1)
    embedding_maps = {
        "tsne": TSNE(n_components=2, perplexity=min(30.0, len(embeddings) - 1),
                     init="pca", learning_rate="auto", random_state=viz_data["seed"]),
        # "isomap": Isomap(n_components=2, n_neighbors=map_neighbors),
        # "spectral": SpectralEmbedding(n_components=2, n_neighbors=map_neighbors,
        #                               affinity="nearest_neighbors", random_state=viz_data["seed"]),
        "mds": MDS(n_components=2, n_init=4, max_iter=300,
                   init="random", eps=1e-6, random_state=viz_data["seed"]),
    }
    computed = {}
    for name, mapper in embedding_maps.items():
        print(f"Computing {name} for {len(embeddings)} samples...", flush=True)
        coordinates = mapper.fit_transform(embeddings)
        if name == "mds" and mapper.n_iter_ >= mapper.max_iter:
            print("MDS reached max_iter; increase max_iter to check convergence.")
        computed[name] = coordinates
        np.savez_compressed(viz_data["directory"] / f"embedding_{name}.npz",
                            coordinates=coordinates, labels=viz_data["labels"],
                            domains=viz_data["domains"], splits=viz_data["splits"],
                            recording=viz_data["recording"], segment_index=viz_data["segment_index"],
                            parameters=json.dumps(mapper.get_params()), seed=viz_data["seed"],
                            max_per_split=-1 if viz_data["max_per_split"] is None else viz_data["max_per_split"])
    projections = computed
else:
    print("Projection skipped.")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Edit layout, colors, markers and legends here. Cached coordinates are reused.
map_titles = {"tsne": "t-SNE", "isomap": "Isomap", "spectral": "Spectral Embedding", "mds": "Metric MDS"}
if projections is not None:
    for name, coordinates in projections.items():
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)
        for ax, values, title in zip(axes, (viz_data["labels"], viz_data["domains"], viz_data["splits"]),
                                    (f"Task: {viz_data['task']}", f"Domain: {viz_data['domain_key']}", "Data split")):
            categories = list(dict.fromkeys(values))
            colors = plt.get_cmap("tab20", len(categories))
            for i, category in enumerate(categories):
                selected = values == category
                ax.scatter(*coordinates[selected].T, s=12, alpha=0.65,
                           color=colors(i), label=category, linewidths=0)
            ax.set(title=title, xlabel=f"{map_titles[name]} 1")
            ax.xaxis.set_major_locator(MaxNLocator(4))
            ax.legend(fontsize=8, markerscale=1.5)
        axes[0].set_ylabel(f"{map_titles[name]} 2")
        fig.suptitle(f"{map_titles[name]} | n={len(coordinates)}")
        fig.tight_layout()
        output_path = viz_data["directory"] / f"embedding_{name}.png"
        fig.savefig(output_path, dpi=180)
        plt.show()
        plt.close(fig)
        print("Saved:", output_path)
else:
    print("No completed projections. Run the loading and computation cells first if needed.")
